# IEEE 39-bus system in DP3Ph and EMT3Ph

This notebook builds both three-phase systems directly in Python from the same MATPOWER case and solved power-flow point. Both runs use the same fourth-order synchronous-generator data, network, and load model. The later cells compare initialization, voltages, runtime, and the response to a temporary load step.


In [ ]:
from pathlib import Path
import sys
import time

_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
REPO = next(p for p in _candidates if (p / "python" / "src" / "dpsim").is_dir())
sys.path.insert(0, str(REPO / "build"))
sys.path.insert(0, str(REPO / "python" / "src"))

import dpsimpy

# Preserve the real EMT ph3 breaker before MATPOWER rewrites its Switch alias.
EMT_PH3_SWITCH = dpsimpy.emt.ph3.Switch
import numpy as np
import pandas as pd
import importlib
from dpsim import matpower as _matpower

matpower = importlib.reload(_matpower)
assert (
    Path(matpower.__file__).resolve()
    == (REPO / "python" / "src" / "dpsim" / "matpower.py").resolve()
), matpower.__file__

## MATPOWER case data

The static network follows MATPOWER `case39`. The dynamic table supplies the common fourth-order machine data for the DP3Ph and EMT3Ph cases.


In [ ]:
"""MATPOWER IEEE 39-bus New England case data and DPsim helpers.

The steady-state values follow MATPOWER's case39 (100 MVA base). Generator
dynamic values use the commonly published 10-machine New England data set.
"""

from pathlib import Path
import tempfile

import numpy as np
from scipy.io import savemat


BASE_MVA = 100.0
FREQUENCY = 60.0
BASE_KV = 345.0

# MATPOWER columns: bus type Pd Qd Gs Bs area Vm Va baseKV zone Vmax Vmin
BUS = np.array(
    [
        [1, 1, 97.6, 44.2, 0, 0, 2, 1.0393836, -13.536602, 345, 1, 1.06, 0.94],
        [2, 1, 0, 0, 0, 0, 2, 1.0484941, -9.7852666, 345, 1, 1.06, 0.94],
        [3, 1, 322, 2.4, 0, 0, 2, 1.0307077, -12.276384, 345, 1, 1.06, 0.94],
        [4, 1, 500, 184, 0, 0, 1, 1.00446, -12.626734, 345, 1, 1.06, 0.94],
        [5, 1, 0, 0, 0, 0, 1, 1.0060063, -11.192339, 345, 1, 1.06, 0.94],
        [6, 1, 0, 0, 0, 0, 1, 1.0082256, -10.40833, 345, 1, 1.06, 0.94],
        [7, 1, 233.8, 84, 0, 0, 1, 0.99839728, -12.755626, 345, 1, 1.06, 0.94],
        [8, 1, 522, 176.6, 0, 0, 1, 0.99787232, -13.335844, 345, 1, 1.06, 0.94],
        [9, 1, 6.5, -66.6, 0, 0, 1, 1.038332, -14.178442, 345, 1, 1.06, 0.94],
        [10, 1, 0, 0, 0, 0, 1, 1.0178431, -8.170875, 345, 1, 1.06, 0.94],
        [11, 1, 0, 0, 0, 0, 1, 1.0133858, -8.9369663, 345, 1, 1.06, 0.94],
        [12, 1, 8.53, 88, 0, 0, 1, 1.000815, -8.9988236, 345, 1, 1.06, 0.94],
        [13, 1, 0, 0, 0, 0, 1, 1.014923, -8.9299272, 345, 1, 1.06, 0.94],
        [14, 1, 0, 0, 0, 0, 1, 1.012319, -10.715295, 345, 1, 1.06, 0.94],
        [15, 1, 320, 153, 0, 0, 3, 1.0161854, -11.345399, 345, 1, 1.06, 0.94],
        [16, 1, 329, 32.3, 0, 0, 3, 1.0325203, -10.033348, 345, 1, 1.06, 0.94],
        [17, 1, 0, 0, 0, 0, 2, 1.0342365, -11.116436, 345, 1, 1.06, 0.94],
        [18, 1, 158, 30, 0, 0, 2, 1.0315726, -11.986168, 345, 1, 1.06, 0.94],
        [19, 1, 0, 0, 0, 0, 3, 1.0501068, -5.4100729, 345, 1, 1.06, 0.94],
        [20, 1, 680, 103, 0, 0, 3, 0.99101054, -6.8211783, 345, 1, 1.06, 0.94],
        [21, 1, 274, 115, 0, 0, 3, 1.0323192, -7.6287461, 345, 1, 1.06, 0.94],
        [22, 1, 0, 0, 0, 0, 3, 1.0501427, -3.1831199, 345, 1, 1.06, 0.94],
        [23, 1, 247.5, 84.6, 0, 0, 3, 1.0451451, -3.3812763, 345, 1, 1.06, 0.94],
        [24, 1, 308.6, -92.2, 0, 0, 3, 1.038001, -9.9137585, 345, 1, 1.06, 0.94],
        [25, 1, 224, 47.2, 0, 0, 2, 1.0576827, -8.3692354, 345, 1, 1.06, 0.94],
        [26, 1, 139, 17, 0, 0, 2, 1.0525613, -9.4387696, 345, 1, 1.06, 0.94],
        [27, 1, 281, 75.5, 0, 0, 2, 1.0383449, -11.362152, 345, 1, 1.06, 0.94],
        [28, 1, 206, 27.6, 0, 0, 3, 1.0503737, -5.9283592, 345, 1, 1.06, 0.94],
        [29, 1, 283.5, 26.9, 0, 0, 3, 1.0501149, -3.1698741, 345, 1, 1.06, 0.94],
        [30, 2, 0, 0, 0, 0, 2, 1.0499, -7.3704746, 345, 1, 1.06, 0.94],
        [31, 3, 9.2, 4.6, 0, 0, 1, 0.982, 0, 345, 1, 1.06, 0.94],
        [32, 2, 0, 0, 0, 0, 1, 0.9841, -0.1884374, 345, 1, 1.06, 0.94],
        [33, 2, 0, 0, 0, 0, 3, 0.9972, -0.19317445, 345, 1, 1.06, 0.94],
        [34, 2, 0, 0, 0, 0, 3, 1.0123, -1.631119, 345, 1, 1.06, 0.94],
        [35, 2, 0, 0, 0, 0, 3, 1.0494, 1.7765069, 345, 1, 1.06, 0.94],
        [36, 2, 0, 0, 0, 0, 3, 1.0636, 4.4684374, 345, 1, 1.06, 0.94],
        [37, 2, 0, 0, 0, 0, 2, 1.0275, -1.5828988, 345, 1, 1.06, 0.94],
        [38, 2, 0, 0, 0, 0, 3, 1.0265, 3.8928177, 345, 1, 1.06, 0.94],
        [39, 2, 1104, 250, 0, 0, 1, 1.03, -14.535256, 345, 1, 1.06, 0.94],
    ],
    dtype=float,
)

# MATPOWER generator rows (standard columns through APF).
_GEN_VALUES = [
    (30, 250, 161.762, 400, 140, 1.0499, 100, 1040),
    (31, 677.871, 221.574, 300, -100, 0.982, 100, 646),
    (32, 650, 206.965, 300, 150, 0.9841, 100, 725),
    (33, 632, 108.293, 250, 0, 0.9972, 100, 652),
    (34, 508, 166.688, 167, 0, 1.0123, 100, 508),
    (35, 650, 210.661, 300, -100, 1.0494, 100, 687),
    (36, 560, 100.165, 240, 0, 1.0636, 100, 580),
    (37, 540, -1.36945, 250, 0, 1.0275, 100, 564),
    (38, 830, 21.7327, 300, -150, 1.0265, 100, 865),
    (39, 1000, 78.4674, 300, -100, 1.03, 100, 1100),
]
GEN = np.array(
    [
        [
            bus,
            pg,
            qg,
            qmax,
            qmin,
            vg,
            mbase,
            1,
            pmax,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
        ]
        for bus, pg, qg, qmax, qmin, vg, mbase, pmax in _GEN_VALUES
    ],
    dtype=float,
)

# MATPOWER branch rows. All buses use 345 kV; nonzero tap ratios are transformer branches.
_BRANCH_VALUES = [
    (1, 2, 0.0035, 0.0411, 0.6987, 0),
    (1, 39, 0.001, 0.025, 0.75, 0),
    (2, 3, 0.0013, 0.0151, 0.2572, 0),
    (2, 25, 0.007, 0.0086, 0.146, 0),
    (2, 30, 0, 0.0181, 0, 1.025),
    (3, 4, 0.0013, 0.0213, 0.2214, 0),
    (3, 18, 0.0011, 0.0133, 0.2138, 0),
    (4, 5, 0.0008, 0.0128, 0.1342, 0),
    (4, 14, 0.0008, 0.0129, 0.1382, 0),
    (5, 6, 0.0002, 0.0026, 0.0434, 0),
    (5, 8, 0.0008, 0.0112, 0.1476, 0),
    (6, 7, 0.0006, 0.0092, 0.113, 0),
    (6, 11, 0.0007, 0.0082, 0.1389, 0),
    (6, 31, 0, 0.025, 0, 1.07),
    (7, 8, 0.0004, 0.0046, 0.078, 0),
    (8, 9, 0.0023, 0.0363, 0.3804, 0),
    (9, 39, 0.001, 0.025, 1.2, 0),
    (10, 11, 0.0004, 0.0043, 0.0729, 0),
    (10, 13, 0.0004, 0.0043, 0.0729, 0),
    (10, 32, 0, 0.02, 0, 1.07),
    (12, 11, 0.0016, 0.0435, 0, 1.006),
    (12, 13, 0.0016, 0.0435, 0, 1.006),
    (13, 14, 0.0009, 0.0101, 0.1723, 0),
    (14, 15, 0.0018, 0.0217, 0.366, 0),
    (15, 16, 0.0009, 0.0094, 0.171, 0),
    (16, 17, 0.0007, 0.0089, 0.1342, 0),
    (16, 19, 0.0016, 0.0195, 0.304, 0),
    (16, 21, 0.0008, 0.0135, 0.2548, 0),
    (16, 24, 0.0003, 0.0059, 0.068, 0),
    (17, 18, 0.0007, 0.0082, 0.1319, 0),
    (17, 27, 0.0013, 0.0173, 0.3216, 0),
    (19, 20, 0.0007, 0.0138, 0, 1.06),
    (19, 33, 0.0007, 0.0142, 0, 1.07),
    (20, 34, 0.0009, 0.018, 0, 1.009),
    (21, 22, 0.0008, 0.014, 0.2565, 0),
    (22, 23, 0.0006, 0.0096, 0.1846, 0),
    (22, 35, 0, 0.0143, 0, 1.025),
    (23, 24, 0.0022, 0.035, 0.361, 0),
    (23, 36, 0.0005, 0.0272, 0, 1.0),
    (25, 26, 0.0032, 0.0323, 0.531, 0),
    (25, 37, 0.0006, 0.0232, 0, 1.025),
    (26, 27, 0.0014, 0.0147, 0.2396, 0),
    (26, 28, 0.0043, 0.0474, 0.7802, 0),
    (26, 29, 0.0057, 0.0625, 1.029, 0),
    (28, 29, 0.0014, 0.0151, 0.249, 0),
    (29, 38, 0.0008, 0.0156, 0, 1.025),
]
BRANCH = np.array(
    [
        [fb, tb, r, x, b, 0, 0, 0, tap, 0, 1, -360, 360]
        for fb, tb, r, x, b, tap in _BRANCH_VALUES
    ],
    dtype=float,
)

# Generator dynamics from the published 10-machine New England data. H is
# converted to the generator MVA base by the paired BaseS values below.
# Columns follow dpsim.matpower.Reader's gen_dyn format.
_DYN = [
    (
        30,
        10000,
        500,
        0.003,
        7.0,
        0.05,
        0.70,
        0.035,
        0.020,
        0.006,
        0.004,
        0.019,
        0.008,
        0.004,
    ),
    (
        31,
        1000,
        30.3,
        0.035,
        6.56,
        0.05,
        1.5,
        0.035,
        0.295,
        0.0697,
        0.05,
        0.282,
        0.17,
        0.05,
    ),
    (
        32,
        1000,
        35.8,
        0.0304,
        5.7,
        0.05,
        1.5,
        0.035,
        0.2495,
        0.0531,
        0.045,
        0.237,
        0.0876,
        0.045,
    ),
    (
        33,
        1000,
        28.6,
        0.0295,
        5.69,
        0.05,
        1.5,
        0.035,
        0.262,
        0.0436,
        0.035,
        0.258,
        0.166,
        0.035,
    ),
    (
        34,
        600,
        26.0,
        0.054,
        5.4,
        0.05,
        0.44,
        0.035,
        0.67,
        0.132,
        0.089,
        0.62,
        0.166,
        0.089,
    ),
    (
        35,
        1000,
        34.8,
        0.0224,
        7.3,
        0.05,
        0.4,
        0.035,
        0.254,
        0.05,
        0.04,
        0.241,
        0.0814,
        0.04,
    ),
    (
        36,
        1000,
        26.4,
        0.0322,
        5.66,
        0.05,
        1.5,
        0.035,
        0.295,
        0.049,
        0.044,
        0.292,
        0.186,
        0.044,
    ),
    (
        37,
        1000,
        24.3,
        0.028,
        6.7,
        0.05,
        0.41,
        0.035,
        0.29,
        0.057,
        0.045,
        0.28,
        0.0911,
        0.045,
    ),
    (
        38,
        1000,
        34.5,
        0.0298,
        4.79,
        0.05,
        1.96,
        0.035,
        0.2106,
        0.057,
        0.045,
        0.205,
        0.0587,
        0.045,
    ),
    # Bus 39's published q-axis data omits T'q0 and gives Xq'' below Xl.
    # Use a short q-axis time constant and a feasible subtransient value for
    # this 6-state DP winding model.
    (
        39,
        1000,
        42.0,
        0.0125,
        10.2,
        0.05,
        0.035,
        0.035,
        0.1,
        0.031,
        0.025,
        0.069,
        0.025,
        0.015,
    ),
]
GEN_DYN = np.array(
    [
        [bus, 4, base_s, h, 0, xl, td, td2, tq, tq2, xd, xdt, xds, xq, xqt, xqs]
        for (bus, base_s, h, xl, td, td2, tq, tq2, xd, xdt, xds, xq, xqt, xqs) in _DYN
    ],
    dtype=float,
)

# Simple DC1 exciters keep the EMT PV-bus voltages regulated during the
# disturbance-free 30 s run. Parameters are intentionally common and modest;
# these are example defaults, not machine-specific benchmark controls.
AVR = np.array(
    [
        [bus, 0.33, 0.1, 46, -0.0435, 0.1, 0.06, 0, 0, 0.46, 1, 0.02, 1, -0.9]
        for bus, *_ in _GEN_VALUES
        if bus != 31
    ],
    dtype=float,
)


def write_matpower_inputs(directory=None):
    """Write temporary .mat inputs for DPsim's existing Matpower reader."""
    if directory is None:
        directory = tempfile.gettempdir()
    directory = Path(directory)
    static_path = directory / "dpsim_ieee39_case.mat"
    dynamic_path = directory / "dpsim_ieee39_dynamic.mat"
    savemat(
        static_path,
        {"mpc": {"baseMVA": BASE_MVA, "bus": BUS, "gen": GEN, "branch": BRANCH}},
    )
    savemat(dynamic_path, {"mpc": {"gen_dyn": GEN_DYN, "avr": AVR}})
    return static_path, dynamic_path


def phase_matrix(value):
    """Return a balanced three-phase diagonal parameter matrix."""
    return np.eye(3, dtype=float) * value


def phase_envelope(vm_pu, angle_deg):
    """Three-phase DP envelope in phase-to-ground peak volts."""
    v_phase_peak = vm_pu * BASE_KV * 1e3 / np.sqrt(3) * np.sqrt(2)
    theta = np.deg2rad(angle_deg)
    return np.array(
        [
            v_phase_peak * np.exp(1j * theta),
            v_phase_peak * np.exp(1j * (theta - 2 * np.pi / 3)),
            v_phase_peak * np.exp(1j * (theta + 2 * np.pi / 3)),
        ],
        dtype=np.complex128,
    ).reshape(3, 1)

## Common power flow


In [ ]:
static_path, dynamic_path = write_matpower_inputs()

pf_reader = matpower.Reader(str(static_path))
pf_reader.create_dpsim_objects(
    matpower.Domain.PF,
    FREQUENCY,
    log_level=dpsimpy.LogLevel.off,
    filter_out_of_service=True,
)
pf_reader.create_dpsim_topology()
pf = dpsimpy.Simulation("IEEE39_Common_PF")
pf.set_system(pf_reader.system)
pf.set_domain(dpsimpy.Domain.SP)
pf.set_solver(dpsimpy.Solver.NRP)
pf.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
pf.do_init_from_nodes_and_terminals(True)
pf.set_time_step(0.01)
pf.set_final_time(0.01)
pf.run()

print(f"Solved common 39-bus PF: {len(pf_reader.dpsimpy_busses_dict)} buses")

## DP three-phase run

This case is created entirely in Python. The MATPOWER reader uses fourth-order VBR generators at all ten generator buses and the common solved power-flow values for initialization.


In [ ]:
dp_reader = matpower.Reader(str(static_path), mpc_dyn_file_path=str(dynamic_path))
dp_reader.create_dpsim_objects(
    matpower.Domain.DP3PH,
    FREQUENCY,
    log_level=dpsimpy.LogLevel.off,
    with_pss=False,
    with_avr=False,
    with_tg=False,
    filter_out_of_service=True,
    generator_model=matpower.GenModel.SG_4ORDER_VBR,
)
dp_reader.create_dpsim_topology()
dp_reader.system.init_with_powerflow(
    systemPF=pf_reader.system, domain=dpsimpy.Domain.DP
)

dp_logger = dpsimpy.Logger("IEEE39_DP3Ph")
for bus_id in [1, 10, 20, 30, 31, 39]:
    dp_logger.log_attribute(
        f"N{bus_id}.V", "v", dp_reader.dpsimpy_busses_dict[f"N{bus_id}"]
    )
dp_sim = dpsimpy.Simulation("IEEE39_DP3Ph")
dp_sim.set_system(dp_reader.system)
dp_sim.set_domain(dpsimpy.Domain.DP)
dp_sim.set_solver(dpsimpy.Solver.MNA)
dp_sim.do_init_from_nodes_and_terminals(True)
dp_sim.set_time_step(1e-3)
dp_sim.set_final_time(30.0)
dp_sim.add_logger(dp_logger)
dp_start = time.perf_counter()
dp_sim.run()
dp_runtime = time.perf_counter() - dp_start

dp_trace = pd.read_csv("logs/IEEE39_DP3Ph.csv", skipinitialspace=True)
print(
    f"DP3Ph: {len(dp_trace)} samples through {dp_trace.time.iloc[-1]:.3f} s; "
    f"wall time={dp_runtime:.2f} s; "
    f"all finite={np.isfinite(dp_trace.iloc[:, 1:].to_numpy()).all()}"
)

## EMT three-phase run

The EMT network is also created directly in Python with the same fourth-order generators. The notebook checks the initial phasor transfer and the first-cycle voltage profile.


In [ ]:
def run_emt3ph():
    emt_reader = matpower.Reader(str(static_path), mpc_dyn_file_path=str(dynamic_path))
    emt_reader.create_dpsim_objects(
        matpower.Domain.EMT,
        FREQUENCY,
        log_level=dpsimpy.LogLevel.off,
        with_pss=False,
        with_avr=False,
        with_tg=False,
        filter_out_of_service=True,
        generator_model=matpower.GenModel.SG_4ORDER_VBR,
    )
    emt_reader.create_dpsim_topology()
    emt_reader.system.init_with_powerflow(
        systemPF=pf_reader.system, domain=dpsimpy.Domain.EMT
    )

    initial_errors = []
    for bus_name, pf_bus in pf_reader.dpsimpy_busses_dict.items():
        v_pf = pf_bus.single_voltage()
        v_emt = emt_reader.dpsimpy_busses_dict[bus_name].initial_single_voltage()
        initial_errors.append(abs(v_emt - v_pf) / max(abs(v_pf), 1.0))
    print(f"PF-to-EMT initial bus-voltage mismatch: {max(initial_errors):.3e} pu")

    logged_buses = [1, 10, 11, 12, 13, 20, 30, 31, 32, 39]
    logger = dpsimpy.Logger("IEEE39_EMT3Ph", True, 10)
    for bus_id in logged_buses:
        logger.log_attribute(
            f"N{bus_id}.V", "v", emt_reader.dpsimpy_busses_dict[f"N{bus_id}"]
        )
    sim = dpsimpy.Simulation("IEEE39_EMT3Ph")
    sim.set_system(emt_reader.system)
    sim.set_domain(dpsimpy.Domain.EMT)
    sim.set_solver(dpsimpy.Solver.MNA)
    sim.do_init_from_nodes_and_terminals(True)
    sim.set_time_step(1e-4)
    sim.set_final_time(30.0)
    sim.do_system_matrix_recomputation(True)
    sim.add_logger(logger)
    emt_start = time.perf_counter()
    sim.run()
    emt_runtime = time.perf_counter() - emt_start

    frame = pd.read_csv("logs/IEEE39_EMT3Ph.csv", skipinitialspace=True)
    period = 1.0 / FREQUENCY
    cycle = frame[(frame.time >= period) & (frame.time < 2 * period)]
    deviations = []
    for bus_id in logged_buses:
        bus_name = f"N{bus_id}"
        values = cycle[[f"{bus_name}.V_{p}" for p in range(3)]].to_numpy()
        measured_pu = np.sqrt(np.mean(values**2)) * np.sqrt(3.0) / (BASE_KV * 1e3)
        target_pu = abs(pf_reader.dpsimpy_busses_dict[bus_name].single_voltage()) / (
            BASE_KV * 1e3
        )
        deviations.append(
            (abs(measured_pu - target_pu), bus_id, measured_pu, target_pu)
        )
    worst = max(deviations)
    print(
        f"EMT: {len(frame)} samples through {frame.time.iloc[-1]:.3f} s; "
        f"wall time={emt_runtime:.2f} s; "
        f"all finite={np.isfinite(frame.iloc[:, 1:].to_numpy()).all()}"
    )
    print(
        f"Largest second-cycle voltage deviation: bus {worst[1]}, "
        f"{worst[2]:.3f} pu vs {worst[3]:.3f} pu "
        f"({100 * worst[0] / worst[3]:.1f}%)"
    )
    return emt_reader, frame, emt_runtime


emt_reader, emt_trace, emt_runtime = run_emt3ph()
print(f"Runtime ratio (EMT3Ph / DP3Ph): {emt_runtime / dp_runtime:.2f}x")

## Voltage magnitudes and initialization comparison

The plots show DP phasor magnitudes and EMT one-cycle RMS magnitudes in per unit, with the solved common power flow as a reference. Both simulations use the same fourth-order generator model. EMT values are cycle averages because its raw instantaneous voltages contain the expected 60 Hz waveform.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

comparison_buses = [1, 10, 20, 30, 31, 39]
dp_base_peak = BASE_KV * 1e3 * np.sqrt(2.0 / 3.0)
phase_base_rms = BASE_KV * 1e3 / np.sqrt(3.0)
period = 1.0 / FREQUENCY

# DP logs contain complex phase peak phasors; average the three balanced phase magnitudes.
dp_magnitudes = {}
for bus_id in comparison_buses:
    phase_values = [
        dp_trace[f"N{bus_id}.V_{phase}.im"].to_numpy()
        + 1j * dp_trace[f"N{bus_id}.V_{phase}.re"].to_numpy()
        for phase in range(3)
    ]
    dp_magnitudes[bus_id] = np.mean(np.abs(phase_values), axis=0) / dp_base_peak


# EMT logs contain instantaneous phase voltages; reduce each cycle to phase RMS per unit.
def emt_cycle_magnitudes(frame, bus_ids):
    times = frame["time"].to_numpy()
    cycle_starts = np.arange(0.0, times[-1] - period + 1e-12, period)
    left = np.searchsorted(times, cycle_starts, side="left")
    right = np.searchsorted(times, cycle_starts + period, side="left")
    centers, magnitudes = [], {bus_id: [] for bus_id in bus_ids}
    for start, first, stop in zip(cycle_starts, left, right):
        if stop - first < int(0.9 * period / np.median(np.diff(times))):
            continue
        centers.append(start + period / 2.0)
        for bus_id in bus_ids:
            phase_rms = [
                np.sqrt(
                    np.mean(frame[f"N{bus_id}.V_{phase}"].to_numpy()[first:stop] ** 2)
                )
                for phase in range(3)
            ]
            magnitudes[bus_id].append(np.mean(phase_rms) / phase_base_rms)
    return np.asarray(centers), {
        bus: np.asarray(values) for bus, values in magnitudes.items()
    }


emt_cycle_times, emt_magnitudes = emt_cycle_magnitudes(emt_trace, comparison_buses)
pf_magnitudes = {
    bus_id: abs(pf_reader.dpsimpy_busses_dict[f"N{bus_id}"].single_voltage())
    / (BASE_KV * 1e3)
    for bus_id in comparison_buses
}

fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for ax, bus_id in zip(axes.flat, comparison_buses):
    ax.plot(dp_trace.time, dp_magnitudes[bus_id], label="DP3Ph", lw=1.0)
    ax.plot(emt_cycle_times, emt_magnitudes[bus_id], label="EMT3Ph (cycle RMS)", lw=1.0)
    ax.axhline(pf_magnitudes[bus_id], color="black", ls="--", lw=0.8, label="Solved PF")
    ax.set_title(f"Bus {bus_id}")
    ax.set_ylabel("Voltage magnitude (pu)")
    ax.grid(True, alpha=0.3)
axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
axes[0, 0].legend(loc="best")
fig.suptitle("IEEE 39-bus voltage magnitudes — full 30-second runs")
fig.tight_layout()
display(fig)
plt.close(fig)

fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for ax, bus_id in zip(axes.flat, comparison_buses):
    ax.plot(dp_trace.time, dp_magnitudes[bus_id], label="DP3Ph", lw=1.0)
    ax.plot(
        emt_cycle_times,
        emt_magnitudes[bus_id],
        label="EMT3Ph (cycle RMS)",
        lw=1.0,
        marker=".",
        ms=3,
    )
    ax.axhline(pf_magnitudes[bus_id], color="black", ls="--", lw=0.8, label="Solved PF")
    ax.set_xlim(0.0, 0.25)
    ax.set_title(f"Bus {bus_id}")
    ax.set_ylabel("Voltage magnitude (pu)")
    ax.grid(True, alpha=0.3)
axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
axes[0, 0].legend(loc="best")
fig.suptitle("IEEE 39-bus voltage magnitudes — startup detail")
fig.tight_layout()
display(fig)
plt.close(fig)

rows = []
for bus_id in comparison_buses:
    dp = dp_magnitudes[bus_id]
    emt = emt_magnitudes[bus_id]
    target = pf_magnitudes[bus_id]
    steady = emt_cycle_times >= 0.5
    rows.append(
        {
            "bus": bus_id,
            "PF (pu)": target,
            "DP t=0 error (%)": 100.0 * (dp[0] / target - 1.0),
            "EMT first-cycle error (%)": 100.0 * (emt[0] / target - 1.0),
            "DP final (pu)": dp[-1],
            "EMT final-cycle (pu)": emt[-1],
            "DP−EMT final (pu)": dp[-1] - emt[-1],
            "EMT steady max-min (pu)": np.ptp(emt[steady]),
        }
    )
comparison = pd.DataFrame(rows).set_index("bus")
display(comparison.round(5))

## Transient comparison: temporary 10% load increase

Both Python-built networks add the same 68 MW resistive load at bus 20 from 1.0 to 1.5 seconds. The plots compare bus voltage recovery and the rotor-speed response of generator 31.


In [ ]:
LOAD_STEP_TIME = 1.0
LOAD_CLEAR_TIME = 1.5
TRANSIENT_FINAL_TIME = 5.0
LOAD_STEP_MW = 0.10 * BUS[19, 2]
LOAD_STEP_RESISTANCE = (BASE_KV * 1e3 * BUS[19, 7]) ** 2 / (LOAD_STEP_MW * 1e6)
# Save the real three-phase switch bindings before Reader configures its aliases.
DP3PH_SWITCH = dpsimpy.dp.ph3.Switch
EMT3PH_SWITCH = EMT_PH3_SWITCH


def run_load_pulse(domain, name, time_step):
    is_dp = domain == matpower.Domain.DP3PH
    reader = matpower.Reader(str(static_path), mpc_dyn_file_path=str(dynamic_path))
    reader.create_dpsim_objects(
        domain,
        FREQUENCY,
        log_level=dpsimpy.LogLevel.off,
        with_pss=False,
        with_avr=False,
        with_tg=False,
        filter_out_of_service=True,
        generator_model=matpower.GenModel.SG_4ORDER_VBR,
    )
    node = reader.dpsimpy_busses_dict["N20"]
    phase_parameter = dpsimpy.Math.single_phase_parameter_to_three_phase
    if is_dp:
        switch = DP3PH_SWITCH("Bus20_LoadStep", dpsimpy.LogLevel.off)
    else:
        switch = EMT3PH_SWITCH("Bus20_LoadStep", dpsimpy.LogLevel.off)
    switch.set_parameters(
        phase_parameter(1e9), phase_parameter(LOAD_STEP_RESISTANCE), False
    )
    ground = dpsimpy.dp.SimNode.gnd if is_dp else dpsimpy.emt.SimNode.gnd
    switch.connect([ground, node])
    reader.dpsimpy_comp_dict["Bus20_LoadStep"] = [switch, [ground, node]]
    reader.create_dpsim_topology()
    reader.system.init_with_powerflow(
        systemPF=pf_reader.system,
        domain=dpsimpy.Domain.DP if is_dp else dpsimpy.Domain.EMT,
    )

    logger = dpsimpy.Logger(name, True, 10 if not is_dp else 1)
    logger.log_attribute("N20.V", "v", node)
    logger.log_attribute("Gen31.w_r", "w_r", reader.dpsimpy_comp_dict["Gen_N31"][0])
    sim = dpsimpy.Simulation(name)
    sim.set_system(reader.system)
    sim.set_domain(dpsimpy.Domain.DP if is_dp else dpsimpy.Domain.EMT)
    sim.set_solver(dpsimpy.Solver.MNA)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_system_matrix_recomputation(True)
    sim.set_time_step(time_step)
    sim.set_final_time(TRANSIENT_FINAL_TIME)
    sim.add_logger(logger)
    sim.add_event(dpsimpy.event.SwitchEvent3Ph(LOAD_STEP_TIME, switch, True))
    sim.add_event(dpsimpy.event.SwitchEvent3Ph(LOAD_CLEAR_TIME, switch, False))
    start = time.perf_counter()
    sim.run()
    runtime = time.perf_counter() - start
    trace = pd.read_csv(f"logs/{name}.csv", skipinitialspace=True)
    return trace, runtime


dp_step_trace, dp_step_runtime = run_load_pulse(
    matpower.Domain.DP3PH, "IEEE39_DP3Ph_LoadPulse", 1e-3
)
emt_step_trace, emt_step_runtime = run_load_pulse(
    matpower.Domain.EMT, "IEEE39_EMT3Ph_LoadPulse", 1e-4
)

# DP phasors are phase-peak complex envelopes; EMT samples are reduced to cycle RMS.
dp_step_phases = [
    dp_step_trace[f"N20.V_{p}.im"].to_numpy()
    + 1j * dp_step_trace[f"N20.V_{p}.re"].to_numpy()
    for p in range(3)
]
dp_step_voltage = np.mean(np.abs(dp_step_phases), axis=0) / dp_base_peak
emt_step_times, emt_step_mags = emt_cycle_magnitudes(emt_step_trace, [20])
emt_step_voltage = emt_step_mags[20]
emt_step_speed = np.interp(
    emt_step_times, emt_step_trace.time, emt_step_trace["Gen31.w_r"]
)
dp_step_speed = dp_step_trace["Gen31.w_r"].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(dp_step_trace.time, dp_step_voltage, label="DP3Ph", lw=1.0)
axes[0].plot(emt_step_times, emt_step_voltage, label="EMT3Ph (cycle RMS)", lw=1.0)
axes[0].set_ylabel("Bus 20 voltage (pu)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].plot(dp_step_trace.time, dp_step_speed, label="DP3Ph", lw=1.0)
axes[1].plot(emt_step_times, emt_step_speed, label="EMT3Ph", lw=1.0)
axes[1].axhline(1.0, color="black", ls="--", lw=0.8)
axes[1].set_ylabel("Generator 31 speed (pu)")
axes[1].set_xlabel("Time (s)")
axes[1].grid(True, alpha=0.3)
for ax in axes:
    ax.axvspan(LOAD_STEP_TIME, LOAD_CLEAR_TIME, color="orange", alpha=0.15)
fig.suptitle("IEEE 39-bus response to the same temporary load increase")
fig.tight_layout()
display(fig)
plt.close(fig)

print(
    f"DP3Ph transient runtime={dp_step_runtime:.2f} s; "
    f"EMT3Ph transient runtime={emt_step_runtime:.2f} s; "
    f"EMT/DP={emt_step_runtime / dp_step_runtime:.2f}x"
)
print(
    f"Bus 20 voltage range: DP {dp_step_voltage.min():.3f}–{dp_step_voltage.max():.3f} pu; "
    f"EMT {emt_step_voltage.min():.3f}–{emt_step_voltage.max():.3f} pu"
)
print(
    f"Generator 31 maximum |speed−1|: DP {np.max(np.abs(dp_step_speed - 1)):.4f} pu; "
    f"EMT {np.max(np.abs(emt_step_speed - 1)):.4f} pu; "
    f"final speeds: DP {dp_step_speed[-1]:.4f}, EMT {emt_step_speed[-1]:.4f} pu"
)